# BS3

---

Let's start with a discussion on what might happen if $\text{accuracy} = 1$ but $ \text{precision} = \text{recall} = \text{F1} = 0$. For a binary classification problem, an accuracy of 100% means:

$\text{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN}=1$

This means:
* Every positive example was correctly predicted positive (TP)
* Every negative example was correctly predicted negative (TN)
* There are no mistakes:
    * FP = 0
    * FN = 0

<u> **So normally, if accuracy is truly 100%, precision, recall, and F1 cannot be zero — assuming the positive class actually exists. Let’s prove this.** </u>

---

Precision:

$$\text{Precision}=\frac{TP}{TP+FP}$$

Recall:

$$\text{Recall}=\frac{TP}{TP+FN}$$

F1:

$$F1=\frac{2PR}{P+R}$$

If accuracy = 100%, then:

$$
FP=FN=0
$$.

This is because if accuracy = 100%, 

$$\frac{TP+TN}{TP+TN+FP+FN}=1$$

For a fraction to equal 1:

$$\text{numerator}=\text{denominator}$$

Therefore:

$$TP+TN = TP+TN+FP+FN$$

Subtract TP+TN from both sides:

$$0=FP+FN$$

Since both FP and FN are counts, they cannot be negative:

$$FP \geq 0$$

$$FN \geq 0$$

The only way their sum can be zero is:

$$\boxed{FP=0,\quad FN=0}$$

Therefore:

$$\text{Precision}=\frac{TP}{TP}=1$$

and:

$$\text{Recall}=\frac{TP}{TP}=1$$

so:

$$F1=1$$

Therefore, there is no normal binary confusion matrix where accuracy = 100% and precision/recall/F1 = 0.

---

 <u> **However, there are some edge cases where people observe this behavior.** </u>

### Case 1: The dataset contains only negative examples

Example:

Confusion matrix:

|                 | Predicted Negative | Predicted Positive |
| --------------- | ------------------:| ------------------:| 
| Actual Negative | 100                | 	0               |
| Actual Positive |	0	               | 0                  |

So:

$$
TN = 100,
FP = 0,
FN = 0,
TP = 0
$$

Accuracy:

$$
\frac{TP+TN}{N}
=
\frac{0+100}{100}
=
1
$$

But:

Precision:

$$
\frac{TP}{TP+FP}
=
\frac{0}{0}
$$

Recall:

$$
\frac{TP}{TP+FN}
=
\frac{0}{0}
$$


They are undefined because the model never predicted positive and there were no positive examples. By default, sklearn will set it to 0 if that happens. This is very likely because for most of the variants in HHT VCEP theere is a lack of functional evidence.

So we might see:

$$
\text{Accuracy} =  1.0,
\text{Precision} = 0,
\text{Recall} = 0,
\text{F1} = 0
$$

This is probably the most likely explanation.



In reality, BS3's accuracy is 0.896, which is slightly different from our previous assumption. (In our previous example the accuracy is 1). But the previous example may still give us hint on why accuracy is high but precision, recall, and F1 is 0. Let's start by computing the confusion matrix for BS3.

In [1]:
from sklearn.metrics import confusion_matrix
from pathlib import Path
import sys
import json
import pandas as pd

APPLIES_LABELS = [False, True]

# ----------------- LOAD BS3 SAMPLE ----------------- #
project_root = Path.cwd().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from evaluation.question1 import load_gold_answer_df
from pipeline import _safe_variant_name

gold_csv = project_root / "full_evaluation_dataset.csv"

gold_df = load_gold_answer_df(str(gold_csv))

predictions = []
prediction_dir = project_root / "outputs/batch_20260708_132813"

if not prediction_dir.exists():
    raise FileNotFoundError(
        f"Could not found predictions at {prediction_dir}"
    )

non_empty_count = 0
predictions = []
for file in sorted(prediction_dir.glob("*.json")):
    if file.name == "batch_summary.json":
        continue

    with open(file, "r", encoding="utf-8") as f:
        payload = json.load(f)

    variant = payload.get("variant")
    results = payload.get("results")

    has_bs3 = any(
        result.get("criterion") == "BS3" for result in results 
    )

    if not has_bs3:
        continue

    if not variant:
        continue
    variant = _safe_variant_name(variant)

    if not gold_df.loc[variant, "gold_applied"]:
        continue

    gold_answer = "BS3" in gold_df.loc[variant, "gold_applied"]

    bs3_prediction = next(
        (result for result in results if result.get("criterion") == "BS3"),
        None
    )

    if not bs3_prediction:
        continue

    pred_answer = bs3_prediction.get("applies")

    if not isinstance(pred_answer, bool):
        continue

    prediction = {
        "variant": variant,
        "criterion": "BS3",
        "evidence": bs3_prediction.get("evidence", "unknown"),
        "reasoning": bs3_prediction.get("reasoning", "unknown"),
        "tool_used": bs3_prediction.get("tool_used", "unknown"),
        "prediction": pred_answer,
        "gold_answer": gold_answer
    }

    predictions.append(prediction)

# ----------------- COMPUTE CONFUSION MATRIX ----------------- #
pred_df = pd.DataFrame(predictions)

answer_df = pred_df[["gold_answer", "prediction"]]

y_true = pred_df["gold_answer"]
y_pred = pred_df["prediction"]
cm = confusion_matrix(
    y_true=y_true,
    y_pred=y_pred,
    labels=APPLIES_LABELS,
)

print("Confusion Matrix (rows=actual applies, cols=predicted applies):")
print("-" * 60)
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {label}" for label in APPLIES_LABELS],
    columns=[f"Pred {label}" for label in APPLIES_LABELS],
)
print(cm_df.to_string())
print("-" * 60)



Confusion Matrix (rows=actual applies, cols=predicted applies):
------------------------------------------------------------
              Pred False  Pred True
Actual False          43          0
Actual True            5          0
------------------------------------------------------------


According to the formula for precision, recall, and F1, 

$$
\text{precision} = \frac{TP}{TP + FP} = \frac{0}{0 + FP} = 0
$$
$$
\text{recall} = \frac{TP}{TP + FN} = \frac{0}{0 + FN} = 0
$$

$$
\text{F1} = \frac{2 * \text{precision} * \text{recall}}{\text{precision} + \text{recall}} = 0
$$



### **Conclusion**
So, as we can see from the confusion matrix and from the example. The root cause of high accuracy but zero precision and recall is that **there is no true positive**. As we've already seen, this is not due to a error in our calculation, and we should not worry about it.

---

# PS3

We run the same script against PS3 to see if it is caused by the same problem

In [2]:
from sklearn.metrics import confusion_matrix, accuracy_score
from pathlib import Path
import sys
import json
import pandas as pd

APPLIES_LABELS = [False, True]

# ----------------- LOAD BS3 SAMPLE ----------------- #
project_root = Path.cwd().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from evaluation.question1 import load_gold_answer_df
from pipeline import _safe_variant_name

gold_csv = project_root / "full_evaluation_dataset.csv"

gold_df = load_gold_answer_df(str(gold_csv))

predictions = []
prediction_dir = project_root / "outputs/batch_20260708_132813"

if not prediction_dir.exists():
    raise FileNotFoundError(
        f"Could not found predictions at {prediction_dir}"
    )

non_empty_count = 0
predictions = []
for file in sorted(prediction_dir.glob("*.json")):
    if file.name == "batch_summary.json":
        continue

    with open(file, "r", encoding="utf-8") as f:
        payload = json.load(f)

    variant = payload.get("variant")
    results = payload.get("results")

    has_ps3 = any(
        result.get("criterion") == "PS3" for result in results 
    )

    if not has_ps3:
        continue

    if not variant:
        continue
    variant = _safe_variant_name(variant)

    if not gold_df.loc[variant, "gold_applied"]:
        continue

    gold_answer = "PS3" in gold_df.loc[variant, "gold_applied"]

    ps3_prediction = next(
        (result for result in results if result.get("criterion") == "PS3"),
        None
    )

    if not ps3_prediction:
        continue

    pred_answer = ps3_prediction.get("applies")

    if not isinstance(pred_answer, bool):
        continue

    prediction = {
        "variant": variant,
        "criterion": "PS3",
        "evidence": ps3_prediction.get("evidence", "unknown"),
        "reasoning": ps3_prediction.get("reasoning", "unknown"),
        "tool_used": ps3_prediction.get("tool_used", "unknown"),
        "prediction": pred_answer,
        "gold_answer": gold_answer
    }

    predictions.append(prediction)

# ----------------- COMPUTE CONFUSION MATRIX ----------------- #
pred_df = pd.DataFrame(predictions)
y_true = pred_df["gold_answer"]
y_pred = pred_df["prediction"]
cm = confusion_matrix(
    y_true=y_true,
    y_pred=y_pred,
    labels=APPLIES_LABELS,
)

print("Confusion Matrix (rows=actual applies, cols=predicted applies):")
print("-" * 60)
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {label}" for label in APPLIES_LABELS],
    columns=[f"Pred {label}" for label in APPLIES_LABELS],
)
print(cm_df.to_string())
print("-" * 60)


Confusion Matrix (rows=actual applies, cols=predicted applies):
------------------------------------------------------------
              Pred False  Pred True
Actual False          40          0
Actual True            8          0
------------------------------------------------------------


### **Conclusion**
So, as we can see from the confusion matrix, **there is no true positive**, and this lead to zero precision, zero recall, and zero F1. As we've already seen, this is not due to a error in our calculation, and we should not worry about it either.

---

# PS1

In [3]:
from sklearn.metrics import confusion_matrix
from pathlib import Path
import sys
import json
import pandas as pd

APPLIES_LABELS = [False, True]

# ----------------- LOAD BS3 SAMPLE ----------------- #
project_root = Path.cwd().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from evaluation.question1 import load_gold_answer_df
from pipeline import _safe_variant_name

gold_csv = project_root / "full_evaluation_dataset.csv"

gold_df = load_gold_answer_df(str(gold_csv))

predictions = []
prediction_dir = project_root / "outputs/batch_20260708_132813"

if not prediction_dir.exists():
    raise FileNotFoundError(
        f"Could not found predictions at {prediction_dir}"
    )

non_empty_count = 0
predictions = []
criterions = []
for file in sorted(prediction_dir.glob("*.json")):
    if file.name == "batch_summary.json":
        continue

    with open(file, "r", encoding="utf-8") as f:
        payload = json.load(f)

    variant = payload.get("variant")
    results = payload.get("results")

    for result in results:
        if result['criterion'] not in criterions:
            criterions.append(result['criterion'])

    has_ps1_supporting = any(
        result.get("criterion") == "PS1" for result in results 
    )

    if not has_ps1_supporting:
        continue

    if not variant:
        continue
    variant = _safe_variant_name(variant)

    if not gold_df.loc[variant, "gold_applied"]:
        continue

    gold_answer = "PS1" in gold_df.loc[variant, "gold_applied"]

    ps1_prediction = next(
        (result for result in results if result.get("criterion") == "PS1"),
        None
    )

    if not ps1_prediction:
        continue

    pred_answer = ps1_prediction.get("applies")

    if not isinstance(pred_answer, bool):
        continue

    prediction = {
        "variant": variant,
        "criterion": "PS1",
        "evidence": ps1_prediction.get("evidence", "unknown"),
        "reasoning": ps1_prediction.get("reasoning", "unknown"),
        "tool_used": ps1_prediction.get("tool_used", "unknown"),
        "prediction": pred_answer,
        "gold_answer": gold_answer
    }

    predictions.append(prediction)

# ----------------- COMPUTE CONFUSION MATRIX ----------------- #
pred_df = pd.DataFrame(predictions)

answer_df = pred_df[["gold_answer", "prediction"]]

y_true = pred_df["gold_answer"]
y_pred = pred_df["prediction"]
cm = confusion_matrix(
    y_true=y_true,
    y_pred=y_pred,
    labels=APPLIES_LABELS,
)

print("Confusion Matrix (rows=actual applies, cols=predicted applies):")
print("-" * 60)
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {label}" for label in APPLIES_LABELS],
    columns=[f"Pred {label}" for label in APPLIES_LABELS],
)
print(cm_df.to_string())
print("-" * 60)

Confusion Matrix (rows=actual applies, cols=predicted applies):
------------------------------------------------------------
              Pred False  Pred True
Actual False          37          0
Actual True            1          0
------------------------------------------------------------


### **Conclusion**
So, as we can see from the confusion matrix, **there is no true positive**, and this lead to zero precision, zero recall, and zero F1. As we've already seen, this is not due to a error in our calculation, and we should not worry about it either.

---

# SUMMARY

As we can already see from example and confusion metrices, having high accuracy but zero precision, recall, and F1 is normal, and we shouldn't worry about it too much.